In [8]:
import pandas as pd
import numpy as np
import torch
import logging
import sys
import os
import shutil
from pathlib import Path
from tqdm import tqdm

from gluonts.dataset.common import ListDataset
from gluonts.torch.model.deepar import DeepAREstimator

# ==========================================
# 1. 配置参数
# ==========================================
ARGS = {
    "data_path": '../../data/Alibaba/cpu_util_percent_selected_machines_100_1min_no_time_stamp.parquet',
    "save_dir": '../../experiments/deepar_single_machine', # 修改保存路径以示区分
    
    # 时间设置
    "freq": "1min",
    "pred_len": 6,          
    "context_len": 60,      
    "stride": 1,            
    
    # 训练设置 (每台机器单独训练)
    # 注意：因为要训练100次，建议减少Epoch或Batch数，否则会跑很久
    "epochs": 5,
    "batches_per_epoch": 50, # 单机数据少，可以适当减小
    "num_samples": 100       
}

# --- 配置日志 ---
def setup_logging(save_dir):
    path = Path(save_dir)
    path.mkdir(parents=True, exist_ok=True)
    
    log_path = path / "train_all_machines.log"
    # 如果存在旧日志则删除，保持清爽
    if log_path.exists():
        os.remove(log_path)
        
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s',
        handlers=[
            logging.FileHandler(log_path),
            logging.StreamHandler(sys.stdout)
        ]
    )
    return logging.getLogger()

logger = setup_logging(ARGS['save_dir'])
logger.info(f"配置参数: {ARGS}")
logger.info("模式: Single Machine (每台机器独立训练模型)")

# ==========================================
# 2. 数据加载
# ==========================================
logger.info("正在加载数据...")
data = pd.read_parquet(ARGS['data_path'], engine='pyarrow')

start_time = pd.Period(data['timestamp'].iloc[0], freq=ARGS['freq'])
values = data.drop(columns=['timestamp']).values.T.astype(np.float32)


num_machines, total_len = values.shape
train_len = int(total_len * 0.8)
test_start_idx = train_len
max_idx = total_len - ARGS['pred_len']

logger.info(f"数据加载完成。机器数: {num_machines}, 时间长度: {total_len}")

# ==========================================
# 3. 循环：单机训练 -> 单机预测 -> 单机评估
# ==========================================
total_mse = 0.0
total_mae = 0.0 # 新增
total_crps = 0.0
total_samples_count = 0
# 辅助函数
def calc_metrics(samples, truth):
    # samples: (num_samples, pred_len)
    # truth: (pred_len,)
    
    pred_mean = np.mean(samples, axis=0)
    
    # 1. MSE
    mse = np.mean((pred_mean - truth) ** 2)
    
    # 2. MAE (新增)
    mae = np.mean(np.abs(pred_mean - truth))
    
    # 3. CRPS
    term1 = np.mean(np.abs(samples - truth[None, :]), axis=0)
    term2 = np.mean(np.abs(samples[:, None, :] - samples[None, :, :]), axis=(0, 1)) * 0.5
    crps = np.mean(term1 - term2)
    
    return mse, mae, crps

logger.info("开始逐台机器处理...")

# 外层循环：遍历每一台机器
# 使用 tqdm 显示机器的处理进度
for m_idx in range(num_machines):
    machine_data = values[m_idx, :]
    
    # ----------------------------------------
    # A. 构造该机器的训练集
    # ----------------------------------------
    train_ds = ListDataset(
        [{"start": start_time, "target": machine_data[:train_len]}],
        freq=ARGS['freq']
    )
    
    # ----------------------------------------
    # B. 训练该机器的专属模型
    # ----------------------------------------
    # logger.info(f"机器 {m_idx}: 正在训练...") 
    # 为了防止日志刷屏，我们只在 tqdm 里看进度，或者发生错误时打印
    
    estimator = DeepAREstimator(
        freq=ARGS['freq'],
        prediction_length=ARGS['pred_len'],
        context_length=ARGS['context_len'],
        num_batches_per_epoch=ARGS['batches_per_epoch'],
        trainer_kwargs={
            "max_epochs": ARGS['epochs'],
            "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
            "devices": 1,
            "enable_progress_bar": False, # 关闭内部进度条，防止刷屏
            "logger": False
        }
    )
    
    predictor = estimator.train(train_ds)
    
    # ----------------------------------------
    # C. 保存该机器的模型
    # ----------------------------------------
    machine_save_dir = Path(ARGS['save_dir']) / f"machine_{m_idx}"
    if machine_save_dir.exists():
        shutil.rmtree(machine_save_dir) # 清理旧的
    predictor.serialize(machine_save_dir)

    # ----------------------------------------
    # D. 准备回测数据 (串行构造)
    # ----------------------------------------
    input_list = []
    ground_truths = []
    
    for t in range(test_start_idx, max_idx, ARGS['stride']):
        input_list.append({
            "start": start_time,
            "target": machine_data[:t]
        })
        ground_truths.append(machine_data[t : t + ARGS['pred_len']])
    
    if not input_list:
        continue

    # ----------------------------------------
    # E. 预测与评估
    # ----------------------------------------
    test_ds = ListDataset(input_list, freq=ARGS['freq'])
    forecast_it = predictor.predict(test_ds, num_samples=ARGS['num_samples'])
    forecasts = list(forecast_it)
    
    m_mse = 0.0
    m_mae = 0.0
    m_crps = 0.0
    
    for forecast, gt in zip(forecasts, ground_truths):
        mse_val, mae_val, crps_val = calc_metrics(forecast.samples, gt)
        m_mse += mse_val
        m_mae += mae_val
        m_crps += crps_val
        
    # 累加本台机器的指标到全局
    count_this = len(forecasts)
    total_mse += m_mse
    total_mae += m_mae
    total_crps += m_crps
    total_samples_count += count_this
    
    # 计算当前这台机器的平均指标用于日志展示
    cur_rmse = np.sqrt(m_mse / count_this)
    cur_mae = m_mae / count_this
    cur_crps = m_crps / count_this
    
    logger.info(f"Machine {m_idx:02d} | RMSE: {cur_rmse:.4f} | MAE: {cur_mae:.4f} | CRPS: {cur_crps:.4f}")

# ==========================================
# 4. 全局最终结果
# ==========================================
if total_samples_count > 0:
    final_rmse = np.sqrt(total_mse / total_samples_count)
    final_mae = total_mae / total_samples_count
    final_crps = total_crps / total_samples_count
    
    res_str = (
        "\n" + "="*50 + "\n"
        " FINAL RESULTS (Single Machine Training Mode)\n" + 
        "="*50 + "\n"
        f"Total Samples Processed : {total_samples_count}\n"
        f"RMSE (Mean Squared)     : {final_rmse:.4f}\n"
        f"MAE  (Mean Absolute)    : {final_mae:.4f}\n"
        f"CRPS (Probabilistic)    : {final_crps:.4f}\n" +
        "="*50
    )
    logger.info(res_str)
else:
    logger.error("未生成任何预测结果。")

2026-01-11 19:48:30,058 - 配置参数: {'data_path': '../../data/Alibaba/cpu_util_percent_selected_machines_100_1min_no_time_stamp.parquet', 'save_dir': '../../experiments/deepar_single_machine', 'freq': '1min', 'pred_len': 6, 'context_len': 60, 'stride': 1, 'epochs': 5, 'batches_per_epoch': 50, 'num_samples': 100}
2026-01-11 19:48:30,059 - 模式: Single Machine (每台机器独立训练模型)
2026-01-11 19:48:30,061 - 正在加载数据...


2026-01-11 19:48:30,089 - 数据加载完成。机器数: 100, 时间长度: 11520
2026-01-11 19:48:30,090 - 开始逐台机器处理...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4,5,0,1,2,3]

  | Name  | Type        | Params | Mode  | In sizes                                                     | Out sizes  
---------------------------------------------------------------------------------------------------------------------------
0 | model | DeepARModel | 24.9 K | train | [[1, 1], [1, 1], [1, 241, 6], [1, 241], [1, 241], [1, 6, 6]] | [1, 100, 6]
---------------------------------------------------------------------------------------------------------------------------
24.9 K    Trainable params
0         Non-trainable params
24.9 K    Total params
0.100     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode
Epoch 0, global step 50: 'train_loss' reached 4.31590 (best 4.31590), saving model to '/data4/luobaozhen/calculate_electricity/WUWEN/models/DeepAR/checkpo

NameError: name 'exit' is not defined